# Multi-Dataset F1 Score Evaluation with 5-Fold Cross-Validation
## Ground Truth Performance Metrics for Conference Publication

This notebook evaluates BERT-based language models across multiple text classification datasets
using **5-fold stratified cross-validation** to produce reliable F1 scores.

### Hardware
- **GPU**: NVIDIA RTX 4090 (24GB VRAM)
- **Batch size**: 32 (train) / 64 (eval)
- **No session timeout constraints**

### Experimental Design (aligned with published LMDFit paper)
- **Full datasets** — no sample caps, all available samples used
- **5-fold stratified cross-validation**
- **3 training epochs** (no early stopping)
- **Progress tracking** — saves after every fold for safety
- **Preflight validation** — all models and datasets are downloaded and verified before training starts

### Models (7 BERT variants)
BERT, SciBERT, LegalBERT, FinancialBERT, PharmBERT, BioBERT, ChemicalBERT

### Datasets (7 datasets, domain-relevant first)
1. Financial PhraseBank (3 classes, ~4.8K)
2. SciCite (3 classes, ~11K)
3. PubMed RCT (5 classes, ~20K)
4. ADE Corpus (2 classes, ~23K)
5. Climate Detection (2 classes, ~2.6K)
6. SST-2 (2 classes, ~68K — train+validation only, test labels hidden)
7. IMDB (2 classes, ~50K)


In [ ]:
# ============================================================
# INSTALL / UPGRADE REQUIRED PACKAGES
# ============================================================
# Run this cell FIRST on a fresh environment.
#
# torch>=2.6 is required due to CVE-2025-32434 (torch.load vulnerability).
# datasets must be <4.0 because several datasets (Financial PhraseBank,
# SciCite) still use .py loading scripts removed in datasets 4.0.
#
# Uncomment and run:
# !pip install "torch>=2.6" -q
# !pip install "datasets>=3.0,<4.0" transformers scipy pandas matplotlib seaborn scikit-learn accelerate -q


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoConfig,
    Trainer,
    TrainingArguments,
)
import torch
from datasets import load_dataset, Dataset, concatenate_datasets
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import warnings
import os
import gc
import json as json_lib
import shutil
from datetime import datetime, timedelta
import time

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 10)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Using device: cuda
GPU: NVIDIA GeForce RTX 4090
Available GPU memory: 25.25 GB


In [2]:
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU Memory cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")


GPU Memory allocated: 0.00 GB
GPU Memory cached: 0.00 GB


## 1. Configuration

### Models, Datasets, and Training Setup

**Models**: 7 BERT variants (6 uncased + BioBERT cased)

**Dataset run order** (domain-relevant first):
1. Financial PhraseBank (3 classes, finance)
2. SciCite (3 classes, scientific citations)
3. PubMed RCT (5 classes, medical abstracts)
4. ADE Corpus (2 classes, pharma)
5. Climate Detection (2 classes, environmental)
6. SST-2 (2 classes, general sentiment)
7. IMDB (2 classes, general sentiment)

**Training**: 3 epochs, batch=32, no early stopping, no sample caps.


In [3]:
# ============================================================
# MODELS
# ============================================================
models_config = {
    'BERT': 'google-bert/bert-base-uncased',
    'SciBERT': 'allenai/scibert_scivocab_uncased',
    'LegalBERT': 'nlpaueb/legal-bert-base-uncased',
    'FinancialBERT': 'ahmedrachid/FinancialBERT',
    'PharmBERT': 'Lianglab/PharmBERT-uncased',
    'BioBERT': 'dmis-lab/biobert-v1.1',
    'ChemicalBERT': 'recobo/chemical-bert-uncased',
    'PubMedBERT' : 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
}

# ============================================================
# DATASETS (ordered by priority: domain-relevant first)
# ============================================================
datasets_config = {}

# --- Priority 1: Domain-specific, smaller datasets ---

datasets_config['Financial'] = {
    'source': 'huggingface',
    'path': 'takala/financial_phrasebank',
    'subset': 'sentences_50agree',
    'splits_to_combine': ['train'],
    'text_column': 'sentence',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 128
}

datasets_config['SciCite'] = {
    'source': 'huggingface',
    'path': 'allenai/scicite',
    'subset': None,
    'splits_to_combine': ['train', 'validation', 'test'],
    'text_column': 'string',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 256
}

datasets_config['PubMed_RCT'] = {
    'source': 'huggingface',
    'path': 'armanc/pubmed-rct20k',
    'subset': None,
    'splits_to_combine': ['train', 'validation', 'test'],
    'text_column': 'text',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 256
}

datasets_config['ADE_Corpus'] = {
    'source': 'huggingface',
    'path': 'SetFit/ade_corpus_v2_classification',
    'subset': None,
    'splits_to_combine': ['train', 'test'],
    'text_column': 'text',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 256
}

datasets_config['Climate_Detection'] = {
    'source': 'huggingface',
    'path': 'climatebert/climate_detection',
    'subset': None,
    'splits_to_combine': ['train', 'test'],
    'text_column': 'text',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 256
}

# --- Priority 2: Larger general datasets ---

datasets_config['SST2'] = {
    'source': 'huggingface',
    'path': 'stanfordnlp/sst2',
    'subset': None,
    'splits_to_combine': ['train', 'validation'],  # test labels are hidden (-1)
    'text_column': 'sentence',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 128
}

datasets_config['IMDB'] = {
    'source': 'huggingface',
    'path': 'stanfordnlp/imdb',
    'subset': None,
    'splits_to_combine': ['train', 'test'],
    'text_column': 'text',
    'label_column': 'label',
    'problem_type': 'single_label_classification',
    'max_length': 512
}

# ============================================================
# TRAINING CONFIG
# ============================================================
training_config = {
    'n_folds': 5,
    'batch_size': 32,
    'eval_batch_size': 64,       # Eval can use larger batch (no gradients)
    'learning_rate': 2e-5,
    'num_epochs': 3,
    'warmup_steps': 100,
    'weight_decay': 0.01,
    'seed': 42,
}

# ============================================================
# SESSION RESILIENCE
# ============================================================
PROGRESS_FILE = 'cv_progress.json'
RESULTS_DIR = 'cv_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Configuration loaded!")
print(f"\nDatasets (in run order): {list(datasets_config.keys())}")
print(f"Models: {list(models_config.keys())}")
print(f"Folds: {training_config['n_folds']}")
print(f"Batch size: {training_config['batch_size']} (train) / {training_config['eval_batch_size']} (eval)")
print(f"Epochs: {training_config['num_epochs']}")
print(f"No sample cap — using full datasets")
print(f"No early stopping")
print(f"\nTotal training runs: {len(models_config) * len(datasets_config) * training_config['n_folds']}")

Configuration loaded!

Datasets (in run order): ['Financial', 'SciCite', 'PubMed_RCT', 'ADE_Corpus', 'Climate_Detection', 'SST2', 'IMDB']
Models: ['BERT', 'SciBERT', 'LegalBERT', 'FinancialBERT', 'PharmBERT', 'BioBERT', 'ChemicalBERT', 'PubMedBERT']
Folds: 5
Batch size: 32 (train) / 64 (eval)
Epochs: 3
No sample cap — using full datasets
No early stopping

Total training runs: 280


## 2. Progress Tracking

### How it works:
- `cv_progress.json` stores every completed fold result
- On restart, already-completed model-dataset-fold combos are skipped
- If a run dies mid-fold, only that one fold is re-run
- All results are also saved as per-pair CSVs in `cv_results/`


In [4]:
def load_progress():
    """Load progress from disk. Returns dict of completed results."""
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            progress = json_lib.load(f)
        print(f"Loaded progress: {len(progress.get('fold_results', []))} completed folds")
        return progress
    return {'fold_results': [], 'completed_pairs': []}


def save_progress(progress):
    """Save progress to disk (atomic write)."""
    tmp_file = PROGRESS_FILE + '.tmp'
    with open(tmp_file, 'w') as f:
        json_lib.dump(progress, f, indent=2)
    os.replace(tmp_file, PROGRESS_FILE)  # Atomic on POSIX


def is_fold_completed(progress, model_name, dataset_name, fold_num):
    """Check if a specific fold has already been completed."""
    for r in progress['fold_results']:
        if (r['model'] == model_name and
            r['dataset'] == dataset_name and
            r['fold'] == fold_num):
            return True
    return False


def is_pair_completed(progress, model_name, dataset_name, n_folds):
    """Check if all folds for a model-dataset pair are done."""
    completed_folds = sum(
        1 for r in progress['fold_results']
        if r['model'] == model_name and r['dataset'] == dataset_name
    )
    return completed_folds >= n_folds


def get_fold_results(progress, model_name, dataset_name):
    """Get all completed fold results for a model-dataset pair."""
    return [
        r for r in progress['fold_results']
        if r['model'] == model_name and r['dataset'] == dataset_name
    ]


def get_session_summary(progress):
    """Print a summary of what's been completed."""
    n_folds = training_config['n_folds']
    if not progress['fold_results']:
        print("No previous progress found. Starting fresh.")
        return
    
    pairs = {}
    for r in progress['fold_results']:
        key = f"{r['model']} x {r['dataset']}"
        pairs[key] = pairs.get(key, 0) + 1
    
    total_pairs = len(models_config) * len(datasets_config)
    done_pairs = sum(1 for count in pairs.values() if count >= n_folds)
    total_folds = total_pairs * n_folds
    
    print(f"\n{'='*60}")
    print(f"  PROGRESS SUMMARY")
    print(f"{'='*60}")
    print(f"  Completed folds: {len(progress['fold_results'])} / {total_folds}")
    print(f"  Completed pairs: {done_pairs} / {total_pairs}")
    print(f"{'='*60}")
    for pair, count in sorted(pairs.items()):
        status = '  DONE' if count >= n_folds else f'  {count}/{n_folds}'
        print(f"  {pair}: {status}")
    print()


# Load existing progress
progress = load_progress()
get_session_summary(progress)


No previous progress found. Starting fresh.


## 3. Preflight Validation & Cache

### Downloads and validates ALL models and datasets before training begins

This cell does three things:
1. **Downloads and caches every model** (tokenizer + config) — verifies architecture compatibility
2. **Downloads and caches every dataset** — verifies column names and label validity
3. **Reports pass/fail** for each — so you catch path/name errors before committing to hours of training

After this cell completes, all subsequent HuggingFace calls load from **local cache**,
avoiding rate-limit issues during the multi-day training run.


In [5]:
def preflight_validate_all(models_config, datasets_config):
    """
    Download, cache, and validate all models and datasets before training.
    Returns (model_status, dataset_status, dataset_cache) dicts.
    
    dataset_cache holds the loaded+combined datasets so they don't need
    to be re-downloaded during training.
    """
    print(f"{'='*70}")
    print(f"  PREFLIGHT VALIDATION")
    print(f"  Downloading and verifying all models and datasets...")
    print(f"{'='*70}\n")
    
    model_status = {}
    dataset_status = {}
    dataset_cache = {}  # Stores (full_data, label_info) per dataset
    
    # --- Validate Models ---
    print("--- MODELS ---")
    for model_name, model_path in models_config.items():
        try:
            print(f"  [{model_name}] Loading from {model_path}...", end=' ')
            
            # Download config (tiny, just metadata)
            config = AutoConfig.from_pretrained(model_path)
            
            # Verify architecture
            hidden_size = config.hidden_size
            num_layers = config.num_hidden_layers
            num_heads = config.num_attention_heads
            
            # Download tokenizer (caches locally)
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            vocab_size = tokenizer.vocab_size
            
            # Download model weights (caches locally) — then immediately free
            _ = AutoModelForSequenceClassification.from_pretrained(
                model_path, num_labels=2  # dummy, just to trigger download
            )
            del _
            gc.collect()
            
            status = f"OK (hidden={hidden_size}, layers={num_layers}, heads={num_heads}, vocab={vocab_size})"
            model_status[model_name] = ('PASS', status)
            print(status)
            
        except Exception as e:
            status = f"FAIL: {str(e)}"
            model_status[model_name] = ('FAIL', status)
            print(status)
    
    print()
    
    # --- Validate Datasets ---
    print("--- DATASETS ---")
    for ds_name, ds_config in datasets_config.items():
        try:
            print(f"  [{ds_name}] Loading from {ds_config['path']}...", end=' ')
            
            # Download dataset
            if ds_config.get('subset'):
                raw_ds = load_dataset(ds_config['path'], ds_config['subset'])
            else:
                raw_ds = load_dataset(ds_config['path'])
            
            # Check splits exist
            splits_to_combine = ds_config.get('splits_to_combine', ['train'])
            datasets_to_concat = []
            missing_splits = []
            for split_name in splits_to_combine:
                if split_name in raw_ds:
                    datasets_to_concat.append(raw_ds[split_name])
                else:
                    missing_splits.append(split_name)
            
            if not datasets_to_concat:
                raise ValueError(f"No valid splits found. Available: {list(raw_ds.keys())}")
            
            full_data = concatenate_datasets(datasets_to_concat) if len(datasets_to_concat) > 1 else datasets_to_concat[0]
            
            # Verify columns exist
            text_col = ds_config['text_column']
            label_col = ds_config['label_column']
            if text_col not in full_data.column_names:
                raise ValueError(f"Text column '{text_col}' not found. Available: {full_data.column_names}")
            if label_col not in full_data.column_names:
                raise ValueError(f"Label column '{label_col}' not found. Available: {full_data.column_names}")
            
            # Verify labels are valid (no -1 hidden labels)
            unique_labels = sorted(list(set(full_data[label_col])))
            if -1 in unique_labels:
                # Filter out hidden labels (SST-2 test set issue)
                pre_filter = len(full_data)
                full_data = full_data.filter(lambda x: x[label_col] != -1)
                unique_labels = sorted(list(set(full_data[label_col])))
                print(f"(filtered {pre_filter - len(full_data)} hidden labels) ", end='')
            
            num_labels = len(unique_labels)
            label_counts = Counter(full_data[label_col])
            
            # Build label_info
            label_info = {
                'type': 'single_label',
                'num_labels': num_labels,
                'labels': unique_labels,
                'label2id': {label: idx for idx, label in enumerate(unique_labels)},
                'id2label': {idx: label for idx, label in enumerate(unique_labels)}
            }
            
            # Cache the dataset
            dataset_cache[ds_name] = (full_data, label_info)
            
            warn = f" (missing splits: {missing_splits})" if missing_splits else ""
            status = f"OK — {len(full_data)} samples, {num_labels} classes{warn}"
            dataset_status[ds_name] = ('PASS', status)
            print(status)
            
        except Exception as e:
            dataset_status[ds_name] = ('FAIL', str(e))
            print(f"FAIL: {str(e)}")
    
    # --- Summary ---
    print(f"\n{'='*70}")
    print(f"  PREFLIGHT SUMMARY")
    print(f"{'='*70}")
    
    model_pass = sum(1 for s in model_status.values() if s[0] == 'PASS')
    model_total = len(model_status)
    ds_pass = sum(1 for s in dataset_status.values() if s[0] == 'PASS')
    ds_total = len(dataset_status)
    
    print(f"  Models:   {model_pass}/{model_total} passed")
    print(f"  Datasets: {ds_pass}/{ds_total} passed")
    
    # Print failures prominently
    failures = []
    for name, (result, msg) in model_status.items():
        if result == 'FAIL':
            failures.append(f"    MODEL  {name}: {msg}")
    for name, (result, msg) in dataset_status.items():
        if result == 'FAIL':
            failures.append(f"    DATASET {name}: {msg}")
    
    if failures:
        print(f"\n  FAILURES:")
        for f in failures:
            print(f)
        print(f"\n  Fix the above issues before running training!")
    else:
        print(f"\n  All checks passed! Everything is cached locally.")
        print(f"  The training loop will load from cache (no HF API calls).")
    
    print(f"{'='*70}")
    
    return model_status, dataset_status, dataset_cache


# Run preflight
model_status, dataset_status, dataset_cache = preflight_validate_all(models_config, datasets_config)


  PREFLIGHT VALIDATION

--- MODELS ---
  [BERT] Loading from google-bert/bert-base-uncased... 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


OK (hidden=768, layers=12, heads=12, vocab=30522)
  [SciBERT] Loading from allenai/scibert_scivocab_uncased... 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

OK (hidden=768, layers=12, heads=12, vocab=28895)

--- DATASETS ---
  [Financial] Loading from takala/financial_phrasebank... OK — 4846 samples, 3 classes
  [SciCite] Loading from allenai/scicite... OK — 10969 samples, 3 classes
  [PubMed_RCT] Loading from armanc/pubmed-rct20k... 

Repo card metadata block was not found. Setting CardData to empty.


OK — 235892 samples, 5 classes
  [ADE_Corpus] Loading from SetFit/ade_corpus_v2_classification... 

Repo card metadata block was not found. Setting CardData to empty.


OK — 23516 samples, 2 classes
  [Climate_Detection] Loading from climatebert/climate_detection... OK — 1700 samples, 2 classes
  [SST2] Loading from stanfordnlp/sst2... OK — 68221 samples, 2 classes
  [IMDB] Loading from stanfordnlp/imdb... OK — 50000 samples, 2 classes

  PREFLIGHT SUMMARY
  Models:   8/8 passed
  Datasets: 7/7 passed

  All checks passed! Everything is cached locally.
  The training loop will load from cache (no HF API calls).


## 4. Data Loading Functions

### Full dataset loading — uses preflight cache when available
The training loop will pull from `dataset_cache` (populated during preflight)
instead of re-downloading from HuggingFace.


In [6]:
def load_full_dataset(dataset_name, config, dataset_cache=None, seed=42):
    """
    Load dataset from preflight cache if available, otherwise download.
    Returns: full_dataset, label_info
    """
    # Use cached version from preflight
    if dataset_cache and dataset_name in dataset_cache:
        full_data, label_info = dataset_cache[dataset_name]
        print(f"\n  {dataset_name}: loaded from preflight cache "
              f"({len(full_data)} samples, {label_info['num_labels']} classes)")
        return full_data, label_info
    
    # Fallback: download (shouldn't happen if preflight ran)
    print(f"\nLoading {dataset_name} from HuggingFace (not cached)...")
    
    try:
        if config.get('subset'):
            dataset = load_dataset(config['path'], config['subset'])
        else:
            dataset = load_dataset(config['path'])
        
        splits_to_combine = config.get('splits_to_combine', ['train'])
        datasets_to_concat = []
        for split_name in splits_to_combine:
            if split_name in dataset:
                datasets_to_concat.append(dataset[split_name])
                print(f"  Found split '{split_name}': {len(dataset[split_name])} samples")
            else:
                print(f"  Warning: split '{split_name}' not found, skipping")
        
        if not datasets_to_concat:
            print(f"  ERROR: No valid splits found")
            return None, None
        
        full_data = concatenate_datasets(datasets_to_concat) if len(datasets_to_concat) > 1 else datasets_to_concat[0]
        
        # Filter hidden labels if present
        label_col = config['label_column']
        if -1 in set(full_data[label_col]):
            full_data = full_data.filter(lambda x: x[label_col] != -1)
        
        unique_labels = sorted(list(set(full_data[label_col])))
        num_labels = len(unique_labels)
        label_info = {
            'type': 'single_label',
            'num_labels': num_labels,
            'labels': unique_labels,
            'label2id': {label: idx for idx, label in enumerate(unique_labels)},
            'id2label': {idx: label for idx, label in enumerate(unique_labels)}
        }
        print(f"  {num_labels} classes, {len(full_data)} total samples")
        
        return full_data, label_info
        
    except Exception as e:
        print(f"  ERROR loading {dataset_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None


def get_cv_splits(full_data, label_info, n_folds=5, seed=42):
    """
    Generate stratified train/test index splits for k-fold CV.
    """
    label_col_name = [c for c in full_data.column_names if 'label' in c.lower()][0]
    labels = full_data[label_col_name]
    
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    splits = list(kf.split(np.zeros(len(full_data)), labels))
    return splits


print("Data loading functions defined")

Data loading functions defined


## 5. Tokenization and Dataset Preparation

### Handles text cleaning and label encoding for each fold


In [7]:
def prepare_dataset_for_training(train_data, test_data, tokenizer, config, label_info):
    """
    Tokenize and prepare datasets for training.
    Handles text that might be lists, nested structures, or None values.
    """
    text_col = config['text_column']
    label_col = config['label_column']
    max_length = config.get('max_length', 256)
    
    def clean_text(text):
        """Clean and normalize text to ensure it's a proper string."""
        if text is None:
            return ""
        if isinstance(text, str):
            return text.strip()
        if isinstance(text, (list, np.ndarray, tuple)):
            def flatten(item):
                if isinstance(item, str):
                    return item
                elif isinstance(item, (list, np.ndarray, tuple)):
                    return ' '.join(flatten(sub) for sub in item)
                else:
                    return str(item)
            return flatten(text).strip()
        return str(text).strip()
    
    def tokenize_function(examples):
        texts = examples[text_col]
        if isinstance(texts, list):
            cleaned_texts = [clean_text(t) for t in texts]
        else:
            cleaned_texts = [clean_text(texts)]
        
        tokenized = tokenizer(
            cleaned_texts,
            padding='max_length',
            truncation=True,
            max_length=max_length
        )
        
        # Single-label: map to integer IDs
        tokenized['labels'] = [label_info['label2id'][label] for label in examples[label_col]]
        return tokenized
    
    train_dataset = train_data.map(tokenize_function, batched=True, remove_columns=train_data.column_names)
    test_dataset = test_data.map(tokenize_function, batched=True, remove_columns=test_data.column_names)
    
    train_dataset.set_format('torch')
    test_dataset.set_format('torch')
    
    return train_dataset, test_dataset

print("Tokenization functions defined")


Tokenization functions defined


## 6. Metrics Computation


In [8]:
def compute_metrics(eval_pred, label_info):
    """
    Compute F1 micro and F1 macro metrics for single-label classification.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    f1_micro = f1_score(labels, predictions, average='micro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        'f1_micro': f1_micro,
        'f1_macro': f1_macro
    }

print("Metrics computation defined")


Metrics computation defined


## 7. Training Pipeline (Single Fold)

### Fine-tune model on one fold, evaluate, save result, cleanup

Key design choices:
- **No early stopping** — runs full 3 epochs
- **Batch size 32** (train) / **64** (eval)
- Evaluation at end of each epoch only
- **fp16** mixed precision for speed on 4090
- Models load from HF cache (pre-downloaded in preflight)


In [9]:
def train_and_evaluate_fold(model_name, model_path, dataset_name, dataset_config,
                            train_data, test_data, label_info, training_config, fold_num):
    """
    Fine-tune a model on one fold and evaluate.
    Returns fold result dict, or None on failure.
    """
    n_folds = training_config['n_folds']
    print(f"  Fold {fold_num + 1}/{n_folds}  "
          f"(train={len(train_data)}, test={len(test_data)})")
    
    fold_start = time.time()
    
    try:
        # These load from local cache (downloaded during preflight)
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        train_prepared, test_prepared = prepare_dataset_for_training(
            train_data, test_data, tokenizer, dataset_config, label_info
        )
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            num_labels=label_info['num_labels'],
            id2label=label_info['id2label'],
            label2id=label_info['label2id']
        )
        model.to(device)
        
        output_dir = f"./results/{model_name}_{dataset_name}_fold{fold_num}"
        
        training_args = TrainingArguments(
            output_dir=output_dir,
            eval_strategy='no',          # We evaluate manually after training
            save_strategy='no',           # No checkpoints needed (3 epochs, no early stopping)
            learning_rate=training_config['learning_rate'],
            per_device_train_batch_size=training_config['batch_size'],
            per_device_eval_batch_size=training_config['eval_batch_size'],
            num_train_epochs=training_config['num_epochs'],
            weight_decay=training_config['weight_decay'],
            warmup_steps=training_config['warmup_steps'],
            push_to_hub=False,
            logging_dir=f'{output_dir}/logs',
            logging_steps=100,
            seed=training_config['seed'],
            report_to='none',
            fp16=True,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
        )
        
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_prepared,
            eval_dataset=test_prepared,
            compute_metrics=lambda eval_pred: compute_metrics(eval_pred, label_info),
        )
        
        # Remove NotebookProgressCallback — it crashes when trainer.evaluate()
        # is called after trainer.train() in Jupyter because on_train_end
        # cleans up the progress widget that on_evaluate still expects.
        try:
            from transformers.utils.notebook import NotebookProgressCallback
            trainer.remove_callback(NotebookProgressCallback)
        except (ImportError, ValueError):
            pass  # Not in a notebook or callback not present
        
        train_result = trainer.train()
        eval_result = trainer.evaluate()
        
        fold_elapsed = time.time() - fold_start
        
        fold_result = {
            'model': model_name,
            'dataset': dataset_name,
            'fold': fold_num,
            'f1_micro': eval_result['eval_f1_micro'],
            'f1_macro': eval_result['eval_f1_macro'],
            'eval_loss': eval_result['eval_loss'],
            'train_samples': len(train_prepared),
            'test_samples': len(test_prepared),
            'training_time': train_result.metrics['train_runtime'],
            'total_fold_time': fold_elapsed,
            'timestamp': datetime.now().isoformat(),
        }
        
        print(f"    -> F1 Micro: {fold_result['f1_micro']:.4f}  "
              f"F1 Macro: {fold_result['f1_macro']:.4f}  "
              f"(train: {fold_result['training_time']:.0f}s, "
              f"total: {fold_elapsed:.0f}s)")
        
        # Cleanup: free GPU and disk
        del model, trainer, train_prepared, test_prepared
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        
        return fold_result
        
    except Exception as e:
        print(f"    ERROR in fold {fold_num}: {str(e)}")
        import traceback
        traceback.print_exc()
        
        # Cleanup on error too
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        output_dir = f"./results/{model_name}_{dataset_name}_fold{fold_num}"
        if os.path.exists(output_dir):
            shutil.rmtree(output_dir)
        
        return None

print("Training pipeline defined")


Training pipeline defined


## 8. Main Cross-Validation Loop

### The main loop:
1. For each dataset, load from preflight cache (no HF API call)
2. Run all 7 models against it, skipping completed folds
3. After each fold completes, save progress to disk immediately
4. Prints running ETA and progress stats

**Loop order**: dataset-first (load dataset once, run all models).


In [10]:
def format_time(seconds):
    """Format seconds into human-readable string."""
    if seconds < 60:
        return f"{seconds:.0f}s"
    elif seconds < 3600:
        return f"{seconds/60:.1f}m"
    else:
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        return f"{h}h {m}m"


def run_all_cv(models_config, datasets_config, training_config, progress, dataset_cache=None):
    """
    Main loop: dataset-first ordering.
    Loads each dataset from preflight cache, then runs all models on it.
    Skips anything already completed.
    """
    n_folds = training_config['n_folds']
    total_pairs = len(models_config) * len(datasets_config)
    total_folds = total_pairs * n_folds
    session_start = time.time()
    folds_this_session = 0
    
    pair_num = 0
    
    for dataset_name, dataset_config in datasets_config.items():
        
        # Check if ALL models are done for this dataset
        all_done_for_dataset = all(
            is_pair_completed(progress, mn, dataset_name, n_folds)
            for mn in models_config.keys()
        )
        if all_done_for_dataset:
            pair_num += len(models_config)
            print(f"\n[Dataset: {dataset_name}] ALL MODELS DONE — skipping")
            continue
        
        # Load dataset from preflight cache (no HF API call)
        full_data, label_info = load_full_dataset(
            dataset_name, dataset_config,
            dataset_cache=dataset_cache,
            seed=training_config['seed']
        )
        
        if full_data is None:
            print(f"  Skipping {dataset_name} (load failed)")
            pair_num += len(models_config)
            continue
        
        # Create CV splits (deterministic given seed)
        cv_splits = get_cv_splits(
            full_data, label_info,
            n_folds=n_folds, seed=training_config['seed']
        )
        
        for model_name, model_path in models_config.items():
            pair_num += 1
            
            # Skip if fully completed
            if is_pair_completed(progress, model_name, dataset_name, n_folds):
                print(f"\n[{pair_num}/{total_pairs}] {model_name} x {dataset_name}: "
                      f"ALREADY DONE (skipping)")
                continue
            
            # Progress header
            done_folds = len(progress['fold_results'])
            remaining_folds = total_folds - done_folds
            
            # ETA calculation
            eta_str = "calculating..."
            if folds_this_session > 0:
                elapsed = time.time() - session_start
                avg_fold_time = elapsed / folds_this_session
                eta_seconds = avg_fold_time * remaining_folds
                eta_str = format_time(eta_seconds)
            
            print(f"\n{'='*70}")
            print(f"[{pair_num}/{total_pairs}] {model_name} x {dataset_name}")
            print(f"  Progress: {done_folds}/{total_folds} folds done | "
                  f"Remaining: {remaining_folds} | ETA: {eta_str}")
            print(f"{'='*70}")
            
            # Run each fold
            for fold_num, (train_idx, test_idx) in enumerate(cv_splits):
                if is_fold_completed(progress, model_name, dataset_name, fold_num):
                    print(f"  Fold {fold_num + 1}/{n_folds}: already done (skipping)")
                    continue
                
                train_data = full_data.select(train_idx.tolist())
                test_data = full_data.select(test_idx.tolist())
                
                result = train_and_evaluate_fold(
                    model_name, model_path, dataset_name, dataset_config,
                    train_data, test_data, label_info, training_config, fold_num
                )
                
                if result:
                    # Save immediately after each fold
                    result['num_labels'] = label_info['num_labels']
                    result['total_dataset_samples'] = len(full_data)
                    progress['fold_results'].append(result)
                    save_progress(progress)
                    folds_this_session += 1
                    
                    done_folds = len(progress['fold_results'])
                    print(f"    Progress saved ({done_folds}/{total_folds} folds)")
            
            # After all folds for this pair, save a per-pair CSV
            pair_results = get_fold_results(progress, model_name, dataset_name)
            if pair_results:
                pair_df = pd.DataFrame(pair_results)
                pair_df.to_csv(
                    f"{RESULTS_DIR}/{model_name}_{dataset_name}_folds.csv",
                    index=False
                )
                
                # Print pair summary
                f1_macros = [r['f1_macro'] for r in pair_results]
                f1_micros = [r['f1_micro'] for r in pair_results]
                print(f"  --- {model_name} x {dataset_name} COMPLETE ---")
                print(f"  F1 Macro: {np.mean(f1_macros):.4f} ± {np.std(f1_macros):.4f}")
                print(f"  F1 Micro: {np.mean(f1_micros):.4f} ± {np.std(f1_micros):.4f}")
        
        # Note: we do NOT delete full_data from dataset_cache here,
        # because the cache dict holds the reference. We just let Python
        # manage memory. The big GPU consumers are the models, not datasets.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Session summary
    session_elapsed = time.time() - session_start
    print(f"\n{'='*70}")
    print(f"  SESSION COMPLETE")
    print(f"  Folds completed this session: {folds_this_session}")
    print(f"  Session duration: {format_time(session_elapsed)}")
    if folds_this_session > 0:
        print(f"  Average time per fold: {format_time(session_elapsed / folds_this_session)}")
    print(f"  Total progress: {len(progress['fold_results'])}/{total_folds} folds")
    print(f"{'='*70}")
    
    return progress

print("Main CV loop defined")


Main CV loop defined


## 9. Run Analysis

### Execute the CV loop

**Estimated times on RTX 4090 (batch=32, 3 epochs, full data)**:
- Financial (~4.8K, max_len=128): ~2-4 min per fold
- SciCite (~11K): ~5-10 min per fold
- PubMed RCT (~20K): ~10-20 min per fold
- ADE Corpus (~23K): ~10-20 min per fold
- Climate Detection (~2.6K): ~1-3 min per fold
- SST-2 (~68K, max_len=128): ~20-40 min per fold
- IMDB (~50K, max_len=512): ~40-80 min per fold

**Total**: 7 models × 7 datasets × 5 folds = **245 folds**
**Estimate**: ~40-80 hours total


In [11]:
# ============================================================
# SELECT WHAT TO RUN
# ============================================================

# Option 1: Run everything
models_to_run = models_config
datasets_to_run = datasets_config

# Option 2: Run a subset (uncomment to use)
# models_to_run = {k: v for k, v in models_config.items() if k in ['BERT', 'SciBERT']}
# datasets_to_run = {k: v for k, v in datasets_config.items() if k in ['Financial', 'SciCite']}

# ============================================================
# GO
# ============================================================
print(f"{'='*70}")
print(f"Starting 5-Fold CV Evaluation")
print(f"Models: {list(models_to_run.keys())}")
print(f"Datasets: {list(datasets_to_run.keys())}")
print(f"Batch: {training_config['batch_size']} (train) / {training_config['eval_batch_size']} (eval)")
print(f"Epochs: {training_config['num_epochs']} | Early stopping: OFF")
print(f"Session started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")

progress = run_all_cv(models_to_run, datasets_to_run, training_config, progress,
                      dataset_cache=dataset_cache)

print(f"\nSession ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


Starting 5-Fold CV Evaluation
Models: ['BERT', 'SciBERT', 'LegalBERT', 'FinancialBERT', 'PharmBERT', 'BioBERT', 'ChemicalBERT', 'PubMedBERT']
Datasets: ['Financial', 'SciCite', 'PubMed_RCT', 'ADE_Corpus', 'Climate_Detection', 'SST2', 'IMDB']
Batch: 32 (train) / 64 (eval)
Epochs: 3 | Early stopping: OFF
Session started: 2026-03-09 23:51:48

  Financial: loaded from preflight cache (4846 samples, 3 classes)

[1/56] BERT x Financial
  Progress: 0/280 folds done | Remaining: 280 | ETA: calculating...
  Fold 1/5  (train=3876, test=970)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecat

    -> F1 Micro: 0.8577  F1 Macro: 0.8431  (train: 15s, total: 18s)
    Progress saved (1/280 folds)
  Fold 2/5  (train=3877, test=969)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

    -> F1 Micro: 0.8482  F1 Macro: 0.8283  (train: 54s, total: 57s)
    Progress saved (59/280 folds)
  Fold 5/5  (train=8776, test=2193)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ahmedrachid/FinancialBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

KeyboardInterrupt: 

## 10. Aggregate Results

### Compute mean ± std across folds for each model-dataset pair
You can run this section at any time — it will aggregate whatever has been completed so far.


In [ ]:
def aggregate_cv_results(progress):
    """
    Aggregate fold results into mean ± std per model-dataset pair.
    """
    # Group by model-dataset
    pairs = {}
    for r in progress['fold_results']:
        key = (r['model'], r['dataset'])
        pairs.setdefault(key, []).append(r)
    
    rows = []
    for (model, dataset), folds in sorted(pairs.items()):
        f1_micros = [f['f1_micro'] for f in folds]
        f1_macros = [f['f1_macro'] for f in folds]
        times = [f['training_time'] for f in folds]
        
        rows.append({
            'Model': model,
            'Dataset': dataset,
            'F1_Micro_Mean': np.mean(f1_micros),
            'F1_Micro_Std': np.std(f1_micros),
            'F1_Macro_Mean': np.mean(f1_macros),
            'F1_Macro_Std': np.std(f1_macros),
            'Num_Labels': folds[0].get('num_labels', '?'),
            'Total_Samples': folds[0].get('total_dataset_samples', '?'),
            'Folds_Done': len(folds),
            'Avg_Fold_Time_s': np.mean(times),
            'Total_Time_s': np.sum(times),
        })
    
    df = pd.DataFrame(rows)
    return df


# Aggregate
progress = load_progress()  # Reload in case cells were run out of order
results_df = aggregate_cv_results(progress)

if len(results_df) > 0:
    # Sort by dataset then F1 macro
    results_df = results_df.sort_values(['Dataset', 'F1_Macro_Mean'], ascending=[True, False])
    
    # Save
    results_df.to_csv(f'{RESULTS_DIR}/master_cv_results.csv', index=False)
    print("Master CV results saved to cv_results/master_cv_results.csv")
    print(f"\nCompleted pairs: {len(results_df)} "
          f"(of {len(models_config) * len(datasets_config)} total)")
    
    # Display with formatting
    display_df = results_df.copy()
    display_df['F1 Micro'] = display_df.apply(
        lambda r: f"{r['F1_Micro_Mean']:.4f} ± {r['F1_Micro_Std']:.4f}", axis=1)
    display_df['F1 Macro'] = display_df.apply(
        lambda r: f"{r['F1_Macro_Mean']:.4f} ± {r['F1_Macro_Std']:.4f}", axis=1)
    display(display_df[['Model', 'Dataset', 'F1 Micro', 'F1 Macro', 'Num_Labels', 'Total_Samples', 'Folds_Done']])
    
    # Best model per dataset
    print("\nBest Models per Dataset (by F1 Macro mean):")
    best = results_df.loc[results_df.groupby('Dataset')['F1_Macro_Mean'].idxmax()]
    display(best[['Dataset', 'Model', 'F1_Macro_Mean', 'F1_Macro_Std']])
    
    # Total training time
    total_time = results_df['Total_Time_s'].sum()
    print(f"\nTotal training time so far: {format_time(total_time)}")
else:
    print("No results yet. Run the training loop first.")


## 11. Visualizations

### Heatmap and per-model bar plots (mean ± std)


In [ ]:
def create_f1_heatmap(results_df):
    """
    Create heatmaps showing mean CV F1 scores across all model-dataset combinations.
    """
    if len(results_df) == 0:
        print("No results to visualize")
        return None
    
    models = sorted(results_df['Model'].unique())
    datasets = sorted(results_df['Dataset'].unique())
    
    f1_micro_matrix = np.full((len(models), len(datasets)), np.nan)
    f1_macro_matrix = np.full((len(models), len(datasets)), np.nan)
    
    for _, row in results_df.iterrows():
        i = models.index(row['Model'])
        j = datasets.index(row['Dataset'])
        f1_micro_matrix[i, j] = row['F1_Micro_Mean']
        f1_macro_matrix[i, j] = row['F1_Macro_Mean']
    
    fig, axes = plt.subplots(1, 2, figsize=(24, 8))
    
    for ax, matrix, title in [
        (axes[0], f1_micro_matrix, 'F1 Micro (5-Fold CV Mean)'),
        (axes[1], f1_macro_matrix, 'F1 Macro (5-Fold CV Mean)'),
    ]:
        im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
        ax.set_xticks(range(len(datasets)))
        ax.set_yticks(range(len(models)))
        ax.set_xticklabels(datasets, rotation=45, ha='right')
        ax.set_yticklabels(models)
        ax.set_title(title, fontsize=14, fontweight='bold')
        plt.colorbar(im, ax=ax)
        for i in range(len(models)):
            for j in range(len(datasets)):
                if not np.isnan(matrix[i, j]):
                    ax.text(j, i, f'{matrix[i, j]:.3f}',
                           ha='center', va='center', color='black',
                           fontsize=8, fontweight='bold')
    
    plt.tight_layout()
    return fig


if len(results_df) > 0:
    fig = create_f1_heatmap(results_df)
    if fig:
        plt.savefig(f'{RESULTS_DIR}/cv_f1_heatmap.png', dpi=300, bbox_inches='tight')
        print("Heatmap saved")
        plt.show()


In [ ]:
def plot_model_bars(results_df, model_name):
    """
    Bar plots with error bars for one model across datasets.
    """
    model_data = results_df[results_df['Model'] == model_name].sort_values('Dataset')
    if len(model_data) == 0:
        return None
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    colors = plt.cm.Set3(range(len(model_data)))
    x = range(len(model_data))
    
    for ax, mean_col, std_col, label in [
        (axes[0], 'F1_Micro_Mean', 'F1_Micro_Std', 'F1 Micro'),
        (axes[1], 'F1_Macro_Mean', 'F1_Macro_Std', 'F1 Macro'),
    ]:
        bars = ax.bar(x, model_data[mean_col], yerr=model_data[std_col],
                      color=colors, alpha=0.8, edgecolor='black', capsize=5)
        ax.set_ylabel(label, fontsize=12, fontweight='bold')
        ax.set_title(f'{model_name} - {label} (5-Fold CV)', fontsize=14, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.set_xticks(x)
        ax.set_xticklabels(model_data['Dataset'], rotation=30, ha='right')
        ax.grid(axis='y', alpha=0.3)
        
        for bar, m, s in zip(bars, model_data[mean_col], model_data[std_col]):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + s + 0.02,
                   f'{m:.3f}\n±{s:.3f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    return fig


# Generate per-model plots
if len(results_df) > 0:
    for model_name in results_df['Model'].unique():
        fig = plot_model_bars(results_df, model_name)
        if fig:
            plt.savefig(f'{RESULTS_DIR}/{model_name}_cv_bars.png', dpi=300, bbox_inches='tight')
            plt.show()


## 12. Quick Reference

### Checking progress:
```python
progress = load_progress()
get_session_summary(progress)
```

### If something goes wrong:
- To re-run a specific model-dataset pair, delete its entries from `cv_progress.json`
- To start completely fresh, delete `cv_progress.json`
- Individual fold CSVs are in `cv_results/` as backup

### Starting fresh:
```python
import os
if os.path.exists('cv_progress.json'):
    os.remove('cv_progress.json')
    print('Progress file deleted — starting fresh')
```

### Re-running preflight (if you changed config):
```python
model_status, dataset_status, dataset_cache = preflight_validate_all(models_config, datasets_config)
```

### Note on BioBERT:
BioBERT (`dmis-lab/biobert-v1.1`) is the only **cased** model in this set.
All others are uncased. This is architecturally compatible (same hidden size, layers, heads)
but worth noting in the paper — cased tokenization may affect results on case-sensitive texts.

### Note on package versions:
- `torch>=2.6` is required (CVE-2025-32434 vulnerability fix)
- `datasets>=3.0,<4.0` is required (several datasets use .py scripts removed in 4.0)
